# DSA 4060 - Personalized Movie Recommender
* **Name:** Yahya Mohamed
* **Student ID:** 669622
* **Assigned User ID:** 23

---

## Step 1: Loading and Inspecting the Datasets

To begin the practical quiz, I am loading the movie catalogue (`movies.csv`) and historical ratings dataset (`ratings.csv`) into pandas DataFrames. Before building any recommendation algorithms, it is essential to explore and verify the structural integrity of our data.

In this step, I will inspect the first five rows of each DataFrame, check the total dimensions (rows and columns), and verify whether there are any missing values that might impact our downstream analysis.

In [3]:
from google.colab import files

print("Upload 'movies.csv' and 'ratings.csv' from your local folder:")
uploaded = files.upload()

for fn in uploaded.keys():
    print('Uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))

Upload 'movies.csv' and 'ratings.csv' from your local folder:


Saving movies.csv to movies.csv
Saving ratings.csv to ratings.csv
Uploaded file "movies.csv" with length 1245 bytes
Uploaded file "ratings.csv" with length 4655 bytes


In [4]:
import pandas as pd
import numpy as np

# Load datasets
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

# Display first 5 rows of each dataset
print("=== Movies Dataset (First 5 Rows) ===")
display(movies_df.head())

print("\n=== Ratings Dataset (First 5 Rows) ===")
display(ratings_df.head())

# Show dimensions
print("\n=== Dataset Dimensions ===")
print(f"Movies DataFrame shape  : {movies_df.shape[0]} rows, {movies_df.shape[1]} columns")
print(f"Ratings DataFrame shape : {ratings_df.shape[0]} rows, {ratings_df.shape[1]} columns")

# Check for missing values
print("\n=== Missing Values Check ===")
print("Movies dataset:\n", movies_df.isnull().sum())
print("\nRatings dataset:\n", ratings_df.isnull().sum())

=== Movies Dataset (First 5 Rows) ===


,movie_id,title,genres
0,1,Inception,Sci-Fi|Thriller
1,2,Interstellar,Sci-Fi|Drama
2,3,The Matrix,Sci-Fi|Action
3,4,Arrival,Sci-Fi|Drama
4,5,Edge of Tomorrow,Sci-Fi|Action



=== Ratings Dataset (First 5 Rows) ===


,user_id,movie_id,rating
0,1,2,3.5
1,1,3,4.5
2,1,5,3.5
3,1,6,3.5
4,1,11,3.0



=== Dataset Dimensions ===
Movies DataFrame shape  : 36 rows, 3 columns
Ratings DataFrame shape : 440 rows, 3 columns

=== Missing Values Check ===
Movies dataset:
 movie_id    0
title       0
genres      0
dtype: int64

Ratings dataset:
 user_id     0
movie_id    0
rating      0
dtype: int64


### Task 1 Output Summary

* **Structure & Dimensions:** The `movies.csv` dataset contains 36 rows and 3 columns (`movie_id`, `title`, `genres`), representing our full catalogue. The `ratings.csv` dataset consists of 440 rows and 3 columns (`user_id`, `movie_id`, `rating`), representing historical feedback across users.


* **Data Integrity:** Both datasets have 0 missing values across all attributes. The ratings are numerical scores (e.g., 3.0 to 4.5), and multiple genres are formatted with pipe separators (e.g., `Sci-Fi|Action`). The data is clean and ready for analysis.

## Step 2: Task 2 - Profile Assigned User

Now that my data is clean, I am profiling my assigned user.

Formula: Assigned User ID = (last two digits MOD 40) + 1

My last 2 digits(669622) are 22, 22 MOD 40 = 22,  22+1 = 23

Therefore, **User 23** is my assigned user to analyze

 To build an accurate recommendation model, we must first understand what this specific user enjoys watching.

In this step, I will filter `ratings.csv` for `user_id == 23`, merge it with the movie catalogue, display every movie they have rated, extract their top three highest-rated titles, and analyze their genre preferences based on their highest ratings.

In [5]:
# Filter ratings for assigned User ID 23
user_23_ratings = ratings_df[ratings_df['user_id'] == 23].copy()

# Merge with movies to get titles and genres
user_23_profile = pd.merge(user_23_ratings, movies_df, on='movie_id')
user_23_profile = user_23_profile[['movie_id', 'title', 'genres', 'rating']].sort_values(by='rating', ascending=False)

print("=== All Movies Rated by User 23 ===")
display(user_23_profile)

print("\n=== Top 3 Highest-Rated Movies by User 23 ===")
top_3_movies = user_23_profile.head(3)
display(top_3_movies)

=== All Movies Rated by User 23 ===


,movie_id,title,genres,rating
8,30,The Grand Budapest Hotel,Comedy|Drama,5.0
11,35,Moneyball,Sports|Drama,4.0
3,13,Titanic,Romance|Drama,3.5
4,16,Crazy Rich Asians,Romance|Comedy,3.5
9,32,Jumanji: Welcome to the Jungle,Comedy|Adventure,3.0
10,33,Ford v Ferrari,Sports|Drama,3.0
2,12,Mission: Impossible - Fallout,Action|Thriller,3.0
6,20,Hidden Figures,Drama|Biography,3.0
5,19,The Social Network,Drama|Biography,2.5
0,6,Blade Runner 2049,Sci-Fi|Drama,2.5



=== Top 3 Highest-Rated Movies by User 23 ===


,movie_id,title,genres,rating
8,30,The Grand Budapest Hotel,Comedy|Drama,5.0
11,35,Moneyball,Sports|Drama,4.0
3,13,Titanic,Romance|Drama,3.5


###  Task 2 Output & Genre Preference Analysis

* **Rated Movies Breakdown:** User 23 has rated 12 movies in total, with ratings ranging from a low of 2.0 to a high of 5.0.
* **Top 3 Highest-Rated Movies:**
  1. *The Grand Budapest Hotel* (Rating: 5.0 | Genres: Comedy|Drama)
  2. *Moneyball* (Rating: 4.0 | Genres: Sports|Drama)
  3. *Titanic* (Rating: 3.5 | Genres: Romance|Drama)

* **Genre Preference Analysis:**
  User 23 shows a strong preference for **Drama** and **Comedy**. **Drama** is present across all top three highest-rated films (*The Grand Budapest Hotel*, *Moneyball*, and *Titanic*), as well as 6 out of their 12 total rated movies.
  
  Furthermore, lighthearted narrative elements like **Comedy** feature prominently in their top choice (*The Grand Budapest Hotel*, rated 5.0). Conversely, heavy action and sci-fi/fantasy blockbusters received lower scores (e.g., *Avengers: Endgame* and *Finding Nemo* at 2.0), confirming that character-driven **Drama** and **Comedy** drive their viewing choices.

## Step 3: Task 3 - Build the Recommendation Logic

To construct a personalized content-based recommendation system, I am leveraging the user's explicit feedback. First, I identify positively rated movies by filtering for ratings of 4.0 or higher. These positively rated movies serve as the ground-truth representation of User 23's movie tastes.

Next, I transform the genre metadata into numerical feature vectors using `TfidfVectorizer`. I compute a user preference profile vector by averaging the TF-IDF feature vectors of User 23's liked movies (*The Grand Budapest Hotel* and *Moneyball*). I then compute the cosine similarity between this preference profile and all movies in the catalogue, filter out movies User 23 has already watched, and retrieve the top 5 highest-scoring recommendations.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Filter positively rated movies (rating >= 4.0) for User 23
liked_movies = user_23_profile[user_23_profile['rating'] >= 4.0]
print("=== User 23 Positively Rated Movies (Rating >= 4.0) ===")
display(liked_movies[['movie_id', 'title', 'genres', 'rating']])

# Prepare genre text representation (replace '|' with space)
movies_df['genres_formatted'] = movies_df['genres'].str.replace('|', ' ')

# 2. Vectorize genres using TF-IDF
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(movies_df['genres_formatted'])

# Get indices of liked movies in the original movies_df
liked_indices = movies_df[movies_df['movie_id'].isin(liked_movies['movie_id'])].index

# Compute User 23 preference vector (average vector of liked movies)
user_23_vector = tfidf_matrix[liked_indices].mean(axis=0)
user_23_vector = np.asarray(user_23_vector)

# 3. Compute Cosine Similarity between User 23 profile and all movies
similarity_scores = cosine_similarity(user_23_vector, tfidf_matrix).flatten()
movies_df['similarity_score'] = similarity_scores

# 4. Exclude movies already rated by User 23
rated_movie_ids = user_23_profile['movie_id'].tolist()
unrated_movies = movies_df[~movies_df['movie_id'].isin(rated_movie_ids)].copy()

# Sort by similarity score and pick top 5
top_5_recommendations = unrated_movies.sort_values(by=['similarity_score', 'movie_id'], ascending=[False, True]).head(5)

print("\n=== Top 5 Personalized Movie Recommendations for User 23 ===")
display(top_5_recommendations[['movie_id', 'title', 'genres', 'similarity_score']])

=== User 23 Positively Rated Movies (Rating >= 4.0) ===


,movie_id,title,genres,rating
8,30,The Grand Budapest Hotel,Comedy|Drama,5.0
11,35,Moneyball,Sports|Drama,4.0



=== Top 5 Personalized Movie Recommendations for User 23 ===


,movie_id,title,genres,similarity_score
33,34,Creed,Sports|Drama,0.809673
35,36,Remember the Titans,Sports|Drama,0.809673
17,18,The Shawshank Redemption,Drama,0.689497
30,31,The Hangover,Comedy,0.502571
13,14,The Notebook,Romance|Drama,0.369184


## Step 4: Task 4 - Explain the Recommendations

To make our recommendation system transparent and actionable, I have organized the top five recommended titles into a structured results table. Each recommendation is paired with a clear, data-backed rationale connecting the movie's genre characteristics back to User 23's observed viewing behavior—specifically their high ratings for *The Grand Budapest Hotel* (Comedy|Drama) and *Moneyball* (Sports|Drama).

In [7]:
# Create Task 4 formatted results table
results_data = [
    {
        "Rank": 1,
        "Recommended Movie": "Creed",
        "Genres": "Sports|Drama",
        "Reason": "Exact genre overlap (Sports|Drama) with Moneyball (rated 4.0)."
    },
    {
        "Rank": 2,
        "Recommended Movie": "Remember the Titans",
        "Genres": "Sports|Drama",
        "Reason": "Exact genre overlap (Sports|Drama) with Moneyball (rated 4.0)."
    },
    {
        "Rank": 3,
        "Recommended Movie": "The Shawshank Redemption",
        "Genres": "Drama",
        "Reason": "Shares core Drama genre present across user's top choices (rated 5.0 and 4.0)."
    },
    {
        "Rank": 4,
        "Recommended Movie": "The Hangover",
        "Genres": "Comedy",
        "Reason": "Matches the Comedy genre present in user's top-rated movie, The Grand Budapest Hotel (rated 5.0)."
    },
    {
        "Rank": 5,
        "Recommended Movie": "The Notebook",
        "Genres": "Romance|Drama",
        "Reason": "Combines preferred Drama theme with Romance (similar to user's 3.5-rated Titanic)."
    }
]

recommendations_df = pd.DataFrame(results_data)
print("=== Task 4: Recommendation Results Table ===")
display(recommendations_df)

=== Task 4: Recommendation Results Table ===


,Rank,Recommended Movie,Genres,Reason
0,1,Creed,Sports|Drama,Exact genre overlap (Sports|Drama) with Moneyb...
1,2,Remember the Titans,Sports|Drama,Exact genre overlap (Sports|Drama) with Moneyb...
2,3,The Shawshank Redemption,Drama,Shares core Drama genre present across user's ...
3,4,The Hangover,Comedy,Matches the Comedy genre present in user's top...
4,5,The Notebook,Romance|Drama,Combines preferred Drama theme with Romance (s...


**Interpretation of Task 4 Output Table**

1. Alignment with User Profile.

 Every recommended movie directly aligns with User 23's top preferences. Creed and Remember the Titans rank highest because their genre combination (Sports|Drama) perfectly matches Moneyball (rated 4.0).

2. Genre Coverage.

 The Shawshank Redemption and The Hangover capture the user's primary taste pillars—Drama and Comedy—which were featured in their #1 rated film, The Grand Budapest Hotel (rated 5.0).

3. Exclusion Verification.

 All five recommended titles are unrated by User 23, fulfilling the core requirement while ensuring high contextual relevance.

## Step 5: Task 5 - Critical Thinking

### 1. Limitation of the Recommender
A primary limitation of this content-based recommender is **overspecialization / cold-start feature sparsity**. Because the model relies solely on coarse genre tags, it produces identical similarity scores for movies sharing the same genre combination (e.g., *Creed* and *Remember the Titans* both score 0.81).

It cannot distinguish critical qualitative nuances such as director style, cast, thematic depth, release decade, or narrative tone. Furthermore, it risks trapping the user in a "filter bubble," failing to recommend high-quality films outside their historical genre comfort zone.

### 2. Realistic Improvement
A practical improvement would be to implement a **Hybrid Recommender Model** or enrich the content features:
* **Feature Enrichment:** Incorporate richer text metadata (such as plot synopses, keywords, directors, and actors) vectorized using pre-trained embeddings (e.g., Sentence-BERT or TF-IDF on plot summaries) rather than just broad genre tags.
* **Collaborative Filtering:** Combine content similarity with **Matrix Factorization (SVD)** or **User-User Collaborative Filtering** to incorporate collective intelligence from other similar users. This would allow the system to recommend unexpected, highly rated films that align with broader taste patterns.